# 03 — GPU Inference Evaluation

This notebook benchmarks inference speed for the final trained object detection models on a GPU runtime in Google Colab.

The goal is to measure:

- average inference time per image
- over 20 test images
- on GPU

The notebook uses the final trained weights exported from the local project and stored in Google Drive.

# GPU Inference Benchmarking

This notebook is designed to be executed in **Google Colab**. It evaluates the inference speed (milliseconds per image) of both trained models (YOLOv8n and RT-DETR-l) on the test set.

## Google Colab Setup

1. **Hardware Accelerator**: Go to *Runtime > Change runtime type* and select **T4 GPU**. Inference testing must be done on the GPU to satisfy assignment requirements.

2. **Google Drive Integration**: This notebook requires access to your Google Drive to load the trained weights and test images.

## Expected Drive Structure

Before running this notebook, ensure your Google Drive contains the following structure:

    MyDrive/
    └── object_detection_gpu_eval/
        ├── best_yolo.pt
        ├── best_rtdetr.pt
        └── test_images/

## Benchmark Configuration

* **Benchmark Sample Size**: 20 images
* **Input Resolution**: 640x640 (for both models)
* **Output**: Results will be saved to `gpu_inference_benchmark.csv` in your Drive root.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q ultralytics pandas

In [3]:
from pathlib import Path
import time
import json

import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO, RTDETR

In [4]:
# ===== USER CONFIG =====
DRIVE_ROOT = Path("/content/drive/MyDrive/object_detection_gpu_eval")

YOLO_WEIGHTS = DRIVE_ROOT / "best_yolo.pt"
RTDETR_WEIGHTS = DRIVE_ROOT / "best_rtdetr.pt"
TEST_IMAGES_DIR = DRIVE_ROOT / "test_images"

NUM_BENCHMARK_IMAGES = 20
YOLO_IMG_SIZE = 640
RTDETR_IMG_SIZE = 640

OUTPUT_CSV_PATH = DRIVE_ROOT / "gpu_inference_benchmark.csv"

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device != "cuda":
    raise RuntimeError("GPU runtime is required. In Colab, go to Runtime > Change runtime type > GPU.")

required_paths = [YOLO_WEIGHTS, RTDETR_WEIGHTS, TEST_IMAGES_DIR]
missing_paths = [str(p) for p in required_paths if not p.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Missing required files/folders in Google Drive:\n" + "\n".join(missing_paths)
    )

test_image_paths = sorted(TEST_IMAGES_DIR.glob("*.jpg"))
if len(test_image_paths) < NUM_BENCHMARK_IMAGES:
    raise ValueError(
        f"Expected at least {NUM_BENCHMARK_IMAGES} .jpg images in {TEST_IMAGES_DIR}, "
        f"but found {len(test_image_paths)}"
    )

benchmark_images = test_image_paths[:NUM_BENCHMARK_IMAGES]

print(f"YOLO weights: {YOLO_WEIGHTS}")
print(f"RT-DETR weights: {RTDETR_WEIGHTS}")
print(f"Test images available: {len(test_image_paths)}")
print(f"Benchmark images used: {len(benchmark_images)}")

Device: cuda
YOLO weights: /content/drive/MyDrive/object_detection_gpu_eval/best_yolo.pt
RT-DETR weights: /content/drive/MyDrive/object_detection_gpu_eval/best_rtdetr.pt
Test images available: 20
Benchmark images used: 20


In [6]:
!nvidia-smi

Wed Mar 11 06:14:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
yolo_model = YOLO(str(YOLO_WEIGHTS))
rtdetr_model = RTDETR(str(RTDETR_WEIGHTS))

print("Models loaded successfully.")

Models loaded successfully.


In [8]:
def benchmark_model(model, image_paths, device, imgsz=640, warmup_runs=10, repeat_runs=5):
    """
    Benchmark average per-image inference time in milliseconds on GPU.

    This version benchmarks the whole image set as a batch-like list input
    multiple times after warmup, which is much more stable than timing one
    image at a time through repeated high-level predict() calls.

    Args:
        model: Ultralytics model instance.
        image_paths: List of image paths.
        device: Expected to be 'cuda'.
        imgsz: Inference image size.
        warmup_runs: Number of warmup runs before timing.
        repeat_runs: Number of timed runs over the full image list.

    Returns:
        Dictionary with timing statistics.
    """
    if device != "cuda":
        raise RuntimeError("This benchmark must run on CUDA/GPU.")

    sources = [str(p) for p in image_paths]

    # Warmup
    for _ in range(warmup_runs):
        _ = model.predict(
            source=sources,
            imgsz=imgsz,
            device=device,
            verbose=False,
            stream=False,
        )
        torch.cuda.synchronize()

    run_totals_ms = []

    for _ in range(repeat_runs):
        start = time.perf_counter()

        _ = model.predict(
            source=sources,
            imgsz=imgsz,
            device=device,
            verbose=False,
            stream=False,
        )

        torch.cuda.synchronize()
        end = time.perf_counter()

        elapsed_total_ms = (end - start) * 1000
        run_totals_ms.append(elapsed_total_ms)

    run_totals_ms = np.array(run_totals_ms, dtype=float)
    per_image_ms = run_totals_ms / len(sources)

    return {
        "mean_ms_per_image": float(per_image_ms.mean()),
        "std_ms": float(per_image_ms.std()),
        "min_ms": float(per_image_ms.min()),
        "max_ms": float(per_image_ms.max()),
        "num_images": int(len(sources)),
        "repeat_runs": int(repeat_runs),
    }

In [9]:
yolo_benchmark = benchmark_model(
    model=yolo_model,
    image_paths=benchmark_images,
    device=device,
    imgsz=YOLO_IMG_SIZE,
    warmup_runs=10,
    repeat_runs=5,
)

print("YOLOv8n GPU inference benchmark:")
print(json.dumps(yolo_benchmark, indent=2))

YOLOv8n GPU inference benchmark:
{
  "mean_ms_per_image": 26.642887570003495,
  "std_ms": 3.4039814882152517,
  "min_ms": 20.107394349997776,
  "max_ms": 29.34725040000785,
  "num_images": 20,
  "repeat_runs": 5
}


In [10]:
rtdetr_benchmark = benchmark_model(
    model=rtdetr_model,
    image_paths=benchmark_images,
    device=device,
    imgsz=RTDETR_IMG_SIZE,
    warmup_runs=10,
    repeat_runs=5,
)

print("RT-DETR-l GPU inference benchmark:")
print(json.dumps(rtdetr_benchmark, indent=2))

RT-DETR-l GPU inference benchmark:
{
  "mean_ms_per_image": 64.12695522000149,
  "std_ms": 2.162714839908815,
  "min_ms": 61.68541269998968,
  "max_ms": 67.47841410000319,
  "num_images": 20,
  "repeat_runs": 5
}


In [11]:
benchmark_df = pd.DataFrame([
    {
        "model": "YOLOv8n",
        "mean_ms_per_image": yolo_benchmark["mean_ms_per_image"],
        "std_ms": yolo_benchmark["std_ms"],
        "min_ms": yolo_benchmark["min_ms"],
        "max_ms": yolo_benchmark["max_ms"],
        "num_images": yolo_benchmark["num_images"],
        "device": "GPU",
        "imgsz": YOLO_IMG_SIZE,
    },
    {
        "model": "RT-DETR-l",
        "mean_ms_per_image": rtdetr_benchmark["mean_ms_per_image"],
        "std_ms": rtdetr_benchmark["std_ms"],
        "min_ms": rtdetr_benchmark["min_ms"],
        "max_ms": rtdetr_benchmark["max_ms"],
        "num_images": rtdetr_benchmark["num_images"],
        "device": "GPU",
        "imgsz": RTDETR_IMG_SIZE,
    },
])

benchmark_df.to_csv(OUTPUT_CSV_PATH, index=False)
benchmark_df

,model,mean_ms_per_image,std_ms,min_ms,max_ms,num_images,device,imgsz
0,YOLOv8n,26.642888,3.403981,20.107394,29.347250,20,GPU,640
1,RT-DETR-l,64.126955,2.162715,61.685413,67.478414,20,GPU,640


## Interpretation

The table above reports the average GPU inference time per image for both final models.

This benchmark is intended to support model comparison from both perspectives:

- detection quality
- runtime efficiency

In this project, YOLOv8n is expected to be significantly faster than RT-DETR-l because it is a much lighter one-stage detector, while RT-DETR-l is a larger transformer-based architecture.

In [12]:
yolo_ms = yolo_benchmark["mean_ms_per_image"]
rtdetr_ms = rtdetr_benchmark["mean_ms_per_image"]

print(f"YOLOv8n mean inference time: {yolo_ms:.2f} ms/image")
print(f"RT-DETR-l mean inference time: {rtdetr_ms:.2f} ms/image")

if yolo_ms > 0:
    speedup = rtdetr_ms / yolo_ms
    print(f"RT-DETR-l is approximately {speedup:.2f}x slower than YOLOv8n on this GPU benchmark.")

YOLOv8n mean inference time: 26.64 ms/image
RT-DETR-l mean inference time: 64.13 ms/image
RT-DETR-l is approximately 2.41x slower than YOLOv8n on this GPU benchmark.


In [13]:
summary_text = f"""
GPU inference benchmark over {NUM_BENCHMARK_IMAGES} test images

YOLOv8n:
- mean: {yolo_benchmark['mean_ms_per_image']:.2f} ms/image
- std:  {yolo_benchmark['std_ms']:.2f} ms

RT-DETR-l:
- mean: {rtdetr_benchmark['mean_ms_per_image']:.2f} ms/image
- std:  {rtdetr_benchmark['std_ms']:.2f} ms
""".strip()

print(summary_text)

GPU inference benchmark over 20 test images

YOLOv8n:
- mean: 26.64 ms/image
- std:  3.40 ms

RT-DETR-l:
- mean: 64.13 ms/image
- std:  2.16 ms
